# Local / SSH starter

Select the repository `.venv` kernel after `uv sync --extra notebooks --extra deep-learning`. Run from the project root. Prepare `features-train.csv` and `features-valid.csv` with numeric features and binary species targets for the MLP example. The official train.csv is metadata and cannot be used directly.


In [ ]:
import os

from birdclef.config import LoaderConfig, RuntimeConfig, TrainConfig, get_config
from birdclef.models import BaselineExperimentConfig, BaselineModelConfig
from birdclef.runtime import apply_environment_defaults, get_secret

apply_environment_defaults()
os.environ["BCLEF_DATA__COMPETITION_DIR"] = "data/competition"

RUN_NAME = "baseline"
DATASET_ID = "your-data-v1"
WANDB_OFFLINE = False  # True for submissions
os.environ["BCLEF_TRACKING__OFFLINE"] = str(WANDB_OFFLINE).lower()
if not WANDB_OFFLINE and (api_key := get_secret("WANDB_API_KEY")):
    os.environ["BCLEF_TRACKING__API_KEY"] = api_key

config = get_config(
    BaselineExperimentConfig,
    model=BaselineModelConfig(feature_columns=["feature"], target_columns=["target"]),
    runtime=RuntimeConfig(accelerator="auto", devices="auto"),
    loader=LoaderConfig(batch_size=128, val_batch_size=64, num_workers=0),
    train=TrainConfig(max_epochs=5),
)


In [ ]:
import json
import subprocess
import sys

if config.tracking.api_key is not None:
    os.environ["BCLEF_TRACKING__API_KEY"] = config.tracking.api_key.get_secret_value()

config.data.output_dir.mkdir(parents=True, exist_ok=True)
config_path = config.data.output_dir / f"{RUN_NAME}-config.json"
config_path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")

subprocess.run(
    [
        sys.executable,
        "-m",
        "birdclef.models.baseline.fit",
        "--config",
        str(config_path),
        "--name",
        RUN_NAME,
        "--dataset-id",
        DATASET_ID,
        "--train",
        str(config.data.competition_dir / "features-train.csv"),
        "--valid",
        str(config.data.competition_dir / "features-valid.csv"),
    ],
    check=True,
)
